# Chapter 1: What a domain-specific harness does

Companion to *Agent Harness Engineering* by Praneeth Paikray.

Run all cells from a fresh Python kernel, in order. The notebook uses only the Python standard library and needs no API key, network access, or external data during execution.

You will reproduce the missing-value failure, test criterion boundaries, and distinguish a complete screening packet from a positive eligibility decision. Listing 1.1 matches the book. Later cells expand its exercises.

All patients, protocols, markers, and thresholds are fictional software examples. They have no clinical meaning. The notebook demonstrates deterministic checks, not a model benchmark or a complete clinical application.

## 1. Check the environment

Use Python 3.10 or newer. Jupyter supplies the notebook interface; the chapter code does not import third-party packages.

In [ ]:
import sys

assert sys.version_info >= (3, 10), "Use Python 3.10 or newer."
print("Python", ".".join(map(str, sys.version_info[:3])))

## 2. Reproduce Listing 1.1

Patient P017 is 47 years old. Protocol T004 version 2 requires an age from 18 to 65 and marker M from 3 to 7, including both endpoints. The marker is missing.

The faulty baseline omits the missing measurement, so it reports a pass. The corrected function keeps the marker result as `unknown`. Keep this fixture unchanged. Use the independent cases below to explore other values.

In [ ]:
age, marker = 47, None
# Fault: missing values are omitted.
checks = [18 <= age <= 65]
if marker is not None:
    checks.append(3 <= marker <= 7)
baseline = all(checks)

def criterion(value, lower, upper):
    if value is None:
        return "unknown"
    if lower <= value <= upper:
        return "met"
    return "not met"
results = {
    "age": criterion(age, 18, 65),
    "marker": criterion(marker, 3, 7),
}
assert baseline is True
assert set(results) == {"age", "marker"}
cases = {None: "unknown", 5: "met", 9: "not met"}
for value, expected in cases.items():
    assert criterion(value, 3, 7) == expected
print("Baseline claims pass:", baseline)
print("Criterion results:", results)

The output should include `Baseline claims pass: True` and a marker result of `unknown`. The baseline assertion confirms that the intended defect was reproduced. It does not approve that behavior.

The function assumes finite numeric values in agreed units. `None` represents missing evidence. It does not validate input types, timestamps, permissions, or evidence provenance.

## 3. Check boundaries independently

Both endpoints are included. The cases below cover missing evidence, both endpoints, an interior value, and values below and above the permitted range. Python assertions are exercise checks and may be disabled with `python -O`; they are not production enforcement.

In [ ]:
boundary_cases = [
    (None, "unknown"),
    (2, "not met"),
    (3, "met"),
    (5, "met"),
    (7, "met"),
    (9, "not met"),
]

for value, expected in boundary_cases:
    actual = criterion(value, 3, 7)
    assert actual == expected, (value, expected, actual)
    print(f"marker={str(value):>4}  result={actual}")

print(f"Passed {len(boundary_cases)} marker cases.")

## 4. Compare the acceptance rules

The following baseline generalizes the same omission bug to both fields. It is deliberately wrong and is kept here as a comparison. When every value is missing, its check list is empty and `all([])` is also `True`.

The corrected result keeps one status for each required criterion. It does not turn those statuses into an enrollment decision.

In [ ]:
def faulty_baseline(age_value, marker_value):
    available_checks = []
    if age_value is not None:
        available_checks.append(18 <= age_value <= 65)
    if marker_value is not None:
        available_checks.append(3 <= marker_value <= 7)
    return all(available_checks)


comparison_cases = [
    (47, None, True, "met", "unknown"),
    (47, 5, True, "met", "met"),
    (47, 9, False, "met", "not met"),
    (None, None, True, "unknown", "unknown"),
]

for age_value, marker_value, expected_pass, age_status, marker_status in comparison_cases:
    observed = {
        "age": criterion(age_value, 18, 65),
        "marker": criterion(marker_value, 3, 7),
    }
    claimed_pass = faulty_baseline(age_value, marker_value)
    assert claimed_pass is expected_pass
    assert observed == {"age": age_status, "marker": marker_status}
    print(f"age={age_value}, marker={marker_value}: baseline={claimed_pass}, {observed}")

## 5. Distinguish an absent result from an unknown result

Coverage means every required criterion has a result. The small function below checks only the set of criterion names. It does not validate evidence or decide eligibility.

An absent `age` result fails the coverage check. An explicit `age: unknown` result passes coverage, but a complete packet still needs a reason for the unknown status.

In [ ]:
REQUIRED_CRITERIA = {"age", "marker"}


def check_coverage(criterion_results):
    actual = set(criterion_results)
    if actual != REQUIRED_CRITERIA:
        missing = sorted(REQUIRED_CRITERIA - actual)
        extra = sorted(actual - REQUIRED_CRITERIA)
        raise ValueError(f"Missing criteria: {missing}; extra criteria: {extra}")


check_coverage({"age": "met", "marker": "unknown"})
check_coverage({"age": "unknown", "marker": "unknown"})

for incomplete in [{"marker": "unknown"}, {"age": "met"}, {}]:
    try:
        check_coverage(incomplete)
    except ValueError as error:
        print("Expected rejection:", error)
    else:
        raise AssertionError("An incomplete result was accepted.")

print("Complete criterion sets accepted; incomplete sets rejected.")

## 6. Inspect a completed packet with unresolved evidence

A packet can be ready for review while eligibility remains undetermined. The example records the age evidence separately from the missing marker. The source identifier is a reference to our in-memory fixture, not to a real patient system.

The checks below cover this one authored example. A full validator, permissions, persistence, and tool execution belong to later chapters.

In [ ]:
fixture = {"patient_id": "P017", "age": 47, "marker": None}
packet = {
    "patient_id": fixture["patient_id"],
    "protocol": {"id": "T004", "version": 2},
    "criteria": {
        "age": {
            "status": criterion(fixture["age"], 18, 65),
            "evidence": "fixture:P017:age",
            "reason": "Age 47 is within the inclusive range 18 to 65.",
        },
        "marker": {
            "status": criterion(fixture["marker"], 3, 7),
            "evidence": None,
            "reason": "Marker M is absent from the supplied fixture.",
        },
    },
    "preparation_status": "complete",
    "review_status": "pending",
    "eligibility": "not_determined",
}

check_coverage(packet["criteria"])
assert packet["criteria"]["age"]["status"] == "met"
assert packet["criteria"]["marker"]["status"] == "unknown"
assert packet["criteria"]["marker"]["reason"]
assert packet["preparation_status"] == "complete"
assert packet["review_status"] == "pending"
assert packet["eligibility"] == "not_determined"

from pprint import pprint
pprint(packet, sort_dicts=False)

## 7. Write the task contract

Write a paragraph stating the required output, the treatment of missing evidence, and the action reserved for a human. Decide whether this structured fixture needs an agent loop.

### Suggested answer

Prepare a packet for P017 against T004 version 2, with one result for every criterion. Each `met` or `not met` result must agree with its cited evidence. Each `unknown` result needs a reason. Packet preparation can complete with unresolved evidence, while enrollment remains a human decision. A fixed workflow is sufficient for this fixture.

### What to check in your own domain

Choose a required input that may be missing. Write its expected outcome before adding a model call. Keep missing data separate from access failures and exhausted searches, since those conditions support different conclusions.

The six contracts from the book are task, context, action, state, authority, and evidence. This notebook explores task coverage and evidence status. It does not implement the full harness.

In [ ]:
print("Chapter 1 completed: baseline reproduced, boundaries checked, coverage checked, packet inspected.")

## References

- [Anthropic, Building effective agents](https://www.anthropic.com/engineering/building-effective-agents): workflow and agent distinctions.
- [Yao et al., ReAct](https://arxiv.org/abs/2210.03629): reasoning, actions, and observations.
- [Anthropic, Effective harnesses for long-running agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents): persistent progress and completion checks.

The fictional fixture, code, and exercises in this notebook are original companion material.